# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriKeerthi16-b/Machine-Learning/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
!git clone https://github.com/SriKeerthi16-b/Machine-Learning.git

Cloning into 'Machine-Learning'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 161 (delta 66), reused 98 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 1.88 MiB | 9.36 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [3]:
import os

print(os.listdir("Machine-Learning"))
print(os.path.exists(
    "Machine-Learning/data/raw/content_refresh_anonymized.csv"
))

['work', '.github', 'LICENSE', 'CLAUDE.md', 'SETUP.md', 'outputs', 'AGENTS.md', 'README.md', '.git', 'requirements.txt', 'DATA_USE.md', 'notebooks', 'skills', '.gitignore', 'data', 'submission', 'scripts', 'docs', 'GUIDE.md']
True


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier

DATA_PATH = "Machine-Learning/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Rows: 30000
Columns: 45
Declining rate: 0.542


I chose a Decision Tree because it fits the refresh/content-opportunity lane and is easy to interpret. The goal is to rank pages that are likely to be declining so that the highest-priority pages can be reviewed first. A Decision Tree can capture simple non-linear relationships between signals such as content age, freshness, impressions, position, CTR, and word count while remaining readable. I will compare it with the Week-4 baseline using the same Precision@50 metric.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

model_df = df.dropna(
    subset=feature_cols + ["is_declining_label", "client_id"]
).copy()

X = model_df[feature_cols]
y = model_df["is_declining_label"]
groups = model_df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 17223
Test rows: 5078
Training clients: 25
Test clients: 7
Client overlap: 0


I use a client-grouped holdout split. All pages belonging to a client stay in either the training set or the test set, never both. This is more honest for this question because the model should be evaluated on pages from clients it did not see during training. The split is designed to reduce client-level leakage and better reflect how the model would behave on unseen clients.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

test_scores = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame({
    "actual": y_test.values,
    "score": test_scores
}, index=y_test.index)

results = results.sort_values("score", ascending=False)

top_50 = results.head(50)

model_precision_50 = top_50["actual"].mean()

print("Decision Tree Precision@50:",
      round(model_precision_50, 3))

Decision Tree Precision@50: 0.64


In [8]:
baseline_scores = (
    (model_df.loc[X_test.index, "days_since_last_update"] >= 180)
    &
    (model_df.loc[X_test.index, "impressions_90d"] >= 500)
).astype(int)

baseline_rank = (
    model_df.loc[X_test.index, "impressions_90d"]
    .fillna(0)
    .sort_values(ascending=False)
)

baseline_candidates = baseline_scores.loc[baseline_rank.index]

baseline_precision_50 = (
    model_df.loc[baseline_rank.head(50).index, "is_declining_label"]
    .mean()
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Decision Tree"
    ],
    "Precision@50": [
        baseline_precision_50,
        model_precision_50
    ]
})

comparison

,Method,Precision@50
0,Week-4 baseline,0.42
1,Decision Tree,0.64


I train a small, readable Decision Tree using the same core signals used in the earlier baseline/model experiments. I evaluate the ranking with Precision@50 because the practical question is whether the first 50 pages selected for review contain a high fraction of declining pages. The baseline and model are compared on the same test split and the same metric.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix

predicted = (test_scores >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    predicted
).ravel()

print("True positives:", tp)
print("False positives:", fp)
print("False negatives:", fn)
print("True negatives:", tn)

True positives: 1128
False positives: 788
False negatives: 1520
True negatives: 1642


The model's ranking is useful only if the highest-scored pages are actually declining. I therefore inspect the top-ranked pages and the model's false positives and false negatives. I also inspect feature importance to understand which signals the tree relies on. These results are treated as observed and directional evidence rather than proof of causation.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.